In [1]:
import nltk
nltk.download('movie_reviews') # loads the dataset
nltk.download('punkt')
#!python -m spacy download "en_core_web_sm"


[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [2]:
  from nltk.corpus import movie_reviews
  import random
  import spacy
  from scipy.sparse import coo_matrix, vstack
  import matplotlib.pyplot as plt
  from sklearn.linear_model import LogisticRegression
  from sklearn.feature_extraction.text import CountVectorizer

  nlp_en = spacy.load("en_core_web_sm", disable=['ner', 'parser'])

  documents = [(movie_reviews.raw(fileid), category)
                for category in movie_reviews.categories()
                for fileid in movie_reviews.fileids(category)]

  print("number of docs loaded:", len(documents))

  corpus_raw = [ x[0] for x in documents ]  # corpus_raw is a list of strings (reviews) to be converted into vectors
  y_corpus = [ x[1] for x in documents ]    # y_corpus are the sentiment labels of the reviews (nothing to be done here)
  print(corpus_raw[0])
  print(y_corpus[0])

  random.seed(42)


number of docs loaded: 2000
plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 
one of the guys dies , but his girlfriend continues to see him in her life , and has nightmares . 
what's the deal ? 
watch the movie and " sorta " find out . . . 
critique : a mind-fuck movie for the teen generation that touches on a very cool idea , but presents it in a very bad package . 
which is what makes this review an even harder one to write , since i generally applaud films which attempt to break the mold , mess with your head and such ( lost highway & memento ) , but there are good and bad ways of making all types of films , and these folks just didn't snag this one correctly . 
they seem to have taken this pretty neat concept , but executed it terribly . 
so what are the problems with the movie ? 
well , its main problem is that it's simply too jumbled . 
it starts off " normal " but then downshifts into this " fantasy " world in which you , as an 

In [3]:
import numpy as np
import nltk
from nltk.corpus import movie_reviews
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import spacy

# Load English NLP model
nlp_en = spacy.load("en_core_web_sm", disable=['ner', 'parser'])

In [4]:
# Load movie reviews dataset
documents = [(movie_reviews.raw(fileid), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]

# Extract text and labels
corpus_raw = [x[0] for x in documents]
y_corpus = np.array([1 if x[1] == "pos" else 0 for x in documents])

# Split dataset into Train (70%) and Test (30%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    corpus_raw, y_corpus, test_size=0.3, random_state=42
)

# Further split into Train (75%) and Validation (25%)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

print(f"Train set: {len(X_train)}, Validation set: {len(X_val)}, Test set: {len(X_test)}")



Train set: 1050, Validation set: 350, Test set: 600


In [5]:
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_val_bow = bow_vectorizer.transform(X_val)
X_test_bow = bow_vectorizer.transform(X_test)



In [6]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)
X_test_tfidf = tfidf_vectorizer.transform(X_test)



In [7]:
def spacy_vectorize(texts, nlp_model):
    return np.array([nlp_model(text).vector for text in texts])

X_train_spacy = spacy_vectorize(X_train, nlp_en)
X_val_spacy = spacy_vectorize(X_val, nlp_en)
X_test_spacy = spacy_vectorize(X_test, nlp_en)



In [8]:
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],  # Regularization strength
    "max_iter": [100, 200, 500],   # Number of iterations
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [9]:
def train_and_tune(X_train, X_val, y_train, y_val):
    log_reg = LogisticRegression(random_state=42, class_weight="balanced", solver="liblinear")

    grid_search = GridSearchCV(
        log_reg, param_grid, cv=cv_strategy, scoring="accuracy", n_jobs=-1, verbose=1
    )
    grid_search.fit(X_train, y_train)

    # Evaluate on validation set
    best_model = grid_search.best_estimator_
    y_val_pred = best_model.predict(X_val)
    val_acc = accuracy_score(y_val, y_val_pred)

    print(f"Best Hyperparameters: {grid_search.best_params_}")
    print(f"Validation Accuracy: {val_acc:.4f}")
    print("\nClassification Report:\n", classification_report(y_val, y_val_pred))

    return best_model, val_acc

# Train models for each representation
print("\nTraining BoW Model...")
best_bow_model, acc_bow = train_and_tune(X_train_bow, X_val_bow, y_train, y_val)

print("\nTraining TF-IDF Model...")
best_tfidf_model, acc_tfidf = train_and_tune(X_train_tfidf, X_val_tfidf, y_train, y_val)

print("\nTraining SpaCy Model...")
best_spacy_model, acc_spacy = train_and_tune(X_train_spacy, X_val_spacy, y_train, y_val)



Training BoW Model...
Fitting 5 folds for each of 15 candidates, totalling 75 fits
Best Hyperparameters: {'C': 0.1, 'max_iter': 100}
Validation Accuracy: 0.8314

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.85      0.84       181
           1       0.83      0.82      0.82       169

    accuracy                           0.83       350
   macro avg       0.83      0.83      0.83       350
weighted avg       0.83      0.83      0.83       350


Training TF-IDF Model...
Fitting 5 folds for each of 15 candidates, totalling 75 fits
Best Hyperparameters: {'C': 10, 'max_iter': 100}
Validation Accuracy: 0.8314

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.82      0.83       181
           1       0.82      0.84      0.83       169

    accuracy                           0.83       350
   macro avg       0.83      0.83      0.83       350
weighted avg       0.83  

In [10]:
# Select the best model based on validation accuracy
best_model = None
best_representation = None

if max(acc_bow, acc_tfidf, acc_spacy) == acc_bow:
    best_model = best_bow_model
    best_representation = "BoW"
    X_train_final = bow_vectorizer.fit_transform(X_train_val)
    X_test_final = bow_vectorizer.transform(X_test)
elif max(acc_bow, acc_tfidf, acc_spacy) == acc_tfidf:
    best_model = best_tfidf_model
    best_representation = "TF-IDF"
    X_train_final = tfidf_vectorizer.fit_transform(X_train_val)
    X_test_final = tfidf_vectorizer.transform(X_test)
else:
    best_model = best_spacy_model
    best_representation = "SpaCy"
    X_train_final = spacy_vectorize(X_train_val, nlp_en)
    X_test_final = spacy_vectorize(X_test, nlp_en)

print(f"\nBest Model: {best_representation}")

# Retrain best model on Train + Validation set
best_model.fit(X_train_final, y_train_val)



Best Model: BoW


LogisticRegression(C=0.1, class_weight='balanced', random_state=42,
                   solver='liblinear')

In [11]:
y_test_pred = best_model.predict(X_test_final)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"\nTest Accuracy: {test_accuracy:.4f}")
print("\nFinal Classification Report:\n", classification_report(y_test, y_test_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_test_pred))



Test Accuracy: 0.8183

Final Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.81      0.82       302
           1       0.81      0.82      0.82       298

    accuracy                           0.82       600
   macro avg       0.82      0.82      0.82       600
weighted avg       0.82      0.82      0.82       600


Confusion Matrix:
 [[246  56]
 [ 53 245]]


In [12]:
import pandas as pd

def print_top_tokens(model, vectorizer, top_n=30):
    # Extract feature names (tokens) and model coefficients
    feature_names = vectorizer.get_feature_names_out()
    coefs = model.coef_[0]

    # Combine tokens with their corresponding coefficients
    coef_df = pd.DataFrame({
        'token': feature_names,
        'coefficient': coefs,
        'abs_coefficient': np.abs(coefs)
    })

    # Sort by absolute value to get the most influential tokens
    top_tokens = coef_df.sort_values(by='abs_coefficient', ascending=False).head(top_n)

    print(f"\nTop {top_n} tokens by absolute coefficient weight (BoW):\n")
    print(top_tokens[['token', 'coefficient']].to_string(index=False))

# Call the function with the best BoW model
print_top_tokens(best_bow_model, bow_vectorizer)



Top 30 tokens by absolute coefficient weight (BoW):

        token  coefficient
          bad    -0.518359
         plot    -0.349566
        worst    -0.289644
      nothing    -0.279332
        great     0.273769
         only    -0.260396
          fun     0.259414
         seen     0.256888
unfortunately    -0.255924
         well     0.238851
       boring    -0.238043
          see     0.227296
     director    -0.224894
        looks    -0.223540
         true     0.221039
          any    -0.219829
         most     0.217398
       script    -0.214581
       stupid    -0.210756
         poor    -0.206419
    perfectly     0.205861
       people     0.200164
        quite     0.193649
         many     0.192642
     supposed    -0.191980
    sometimes     0.190590
   especially     0.187433
         shot     0.186573
         lame    -0.186069
 performances     0.184120
